# 실험 결과 분석

모델 × 데이터셋 × 전략 3축 비교  
- **데이터**: `summary.json`, `trajectory_logs.jsonl`, `step_logs.jsonl`
- **비교 축**: 모델(phi35mini / qwen7bcoder), 데이터셋(humaneval / mbpp), 전략(single / repair / code_then_plan / code_then_plan_repair / policy_loop 등)

In [1]:
import json
import os
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 0. 경로 설정

In [2]:
RESULTS_ROOT = Path("../results")

# 존재하는 결과 폴더 자동 탐색
def discover_runs(results_root: Path) -> list[dict]:
    """
    results/<model>/<dataset>/<strategy>/summary.json 구조를 재귀 탐색.
    summary.json 이 있는 폴더를 하나의 run으로 간주.
    """
    runs = []
    for summary_path in sorted(results_root.rglob("summary.json")):
        parts = summary_path.parent.relative_to(results_root).parts
        if len(parts) >= 3:
            model, dataset, strategy = parts[0], parts[1], parts[2]
        elif len(parts) == 2:
            model, dataset, strategy = parts[0], parts[1], "unknown"
        else:
            continue
        runs.append({
            "model": model,
            "dataset": dataset,
            "strategy": strategy,
            "summary_path": summary_path,
            "dir": summary_path.parent,
        })
    return runs

runs = discover_runs(RESULTS_ROOT)

filtered_runs = [
    r for r in runs
    if (r["dataset"] == "humaneval" or r["dataset"] == "mbpp")
    and ("qwen25coder7b" in r["model"] or "phi35mini" in r["model"]) and not "proposed_v3" in r["strategy"] and not "proposed_v2" in r["strategy"] 
]

print(f"Filtered runs: {len(filtered_runs)}")


print(f"발견된 run 수: {len(filtered_runs)}")
for r in filtered_runs:
    print(f"  {r['model']:20s} | {r['dataset']:15s} | {r['strategy']}")

Filtered runs: 20
발견된 run 수: 20
  phi35mini            | humaneval       | code_then_plan
  phi35mini            | humaneval       | code_then_plan_repair
  phi35mini            | humaneval       | proposed_v1
  phi35mini            | humaneval       | repair
  phi35mini            | humaneval       | single
  phi35mini            | mbpp            | code_then_plan
  phi35mini            | mbpp            | code_then_plan_repair
  phi35mini            | mbpp            | proposed_v1
  phi35mini            | mbpp            | repair
  phi35mini            | mbpp            | single
  qwen25coder7b        | humaneval       | code_then_plan
  qwen25coder7b        | humaneval       | code_then_plan_repair
  qwen25coder7b        | humaneval       | proposed_v1
  qwen25coder7b        | humaneval       | repair
  qwen25coder7b        | humaneval       | single
  qwen25coder7b        | mbpp            | code_then_plan
  qwen25coder7b        | mbpp            | code_then_plan_repair
  qwen25cod

## 1. summary.json 로드 → 핵심 지표 비교표

In [3]:
def load_summary(run: dict) -> dict:
    with open(run["summary_path"], encoding="utf-8") as f:
        data = json.load(f)

    row = {
        "model":    run["model"],
        "dataset":  run["dataset"],
        "strategy": run["strategy"],
    }

    # 공통 지표
    row["total_problems"]         = data.get("total_problems", None)
    row["num_success"]            = data.get("num_success", None)
    row["success_at_k"]           = data.get("success_at_k", None)
    row["execution_success_rate"] = data.get("execution_success_rate", None)
    row["conditional_success"]    = data.get("conditional_success", None)
    row["max_calls"]              = data.get("max_calls", None)

    # extra_summary (failure breakdown)
    extra = data.get("extra_summary", {})
    row["code_failed"]         = extra.get("code_failed", None)
    row["define_test_failed"]  = extra.get("define_test_failed", None)
    row["run_test_failed"]     = extra.get("run_test_failed", None)

    # planning stats (code_then_plan_repair 계열)
    ps = data.get("planning_stats", {})
    row["plan_used"]              = ps.get("used_plan", None)
    row["repair_used"]            = ps.get("used_repair", None)
    row["planning_recovery_rate"] = ps.get("planning_recovery_rate", None)
    row["repair_recovery_rate"]   = ps.get("repair_recovery_rate", None)

    # policy_stats (policy_loop 계열)
    pol = data.get("policy_stats", {})
    pl = pol.get("problem_level", {})
    cl = pol.get("call_level", {})
    if pl:
        row["plan_used"]              = pl.get("plan_used_problems", row.get("plan_used"))
        row["repair_used"]            = pl.get("repair_used_problems", row.get("repair_used"))
        row["planning_recovery_rate"] = pl.get("plan_problem_recovery_rate", row.get("planning_recovery_rate"))
        row["repair_recovery_rate"]   = pl.get("repair_problem_recovery_rate", row.get("repair_recovery_rate"))
    if cl:
        row["planning_cycle_count"]        = cl.get("planning_cycle_count", None)
        row["planning_cycle_success_rate"] = cl.get("planning_cycle_success_rate", None)
        row["repair_call_count"]           = cl.get("repair_call_count", None)
        row["repair_call_success_rate"]    = cl.get("repair_call_success_rate", None)

    return row


summary_rows = [load_summary(r) for r in filtered_runs]
df_summary = pd.DataFrame(summary_rows)
df_summary

,model,dataset,strategy,total_problems,num_success,success_at_k,execution_success_rate,conditional_success,max_calls,code_failed,define_test_failed,run_test_failed,plan_used,repair_used,planning_recovery_rate,repair_recovery_rate,planning_cycle_count,planning_cycle_success_rate,repair_call_count,repair_call_success_rate
0,phi35mini,humaneval,code_then_plan,164,128,0.7805,0.9512,0.8205,20,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,phi35mini,humaneval,code_then_plan_repair,164,117,0.7134,0.7378,0.9669,20,0,0,0,162.0000,82.0000,0.7099,0.0122,NaN,NaN,NaN,NaN
2,phi35mini,humaneval,proposed_v1,164,125,0.7622,0.9268,0.8224,20,12,0,27,117.0000,161.0000,0.6667,0.2733,335.0000,0.2328,387.0000,0.1137
3,phi35mini,humaneval,repair,164,3,0.0183,0.0183,1.0000,20,161,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,phi35mini,humaneval,single,164,2,0.0122,0.0366,0.3333,1,158,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,phi35mini,mbpp,code_then_plan,257,200,0.7782,0.8677,0.8969,20,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,phi35mini,mbpp,code_then_plan_repair,257,207,0.8054,0.8755,0.9200,20,0,0,0,92.0000,73.0000,0.4565,0.1233,NaN,NaN,NaN,NaN
7,phi35mini,mbpp,proposed_v1,257,205,0.7977,0.8716,0.9152,20,33,0,19,75.0000,80.0000,0.2267,0.1875,292.0000,0.0582,382.0000,0.0393
8,phi35mini,mbpp,repair,257,179,0.6965,0.7704,0.9040,20,59,0,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,phi35mini,mbpp,single,257,163,0.6342,0.7549,0.8402,1,63,0,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1-1. 핵심 지표만 추린 비교표

In [4]:
KEY_COLS = [
    "model", "dataset", "strategy",
    "total_problems", "num_success",
    "success_at_k", "execution_success_rate", "conditional_success",
]
df_summary[KEY_COLS].sort_values(["dataset", "model", "strategy"])

,model,dataset,strategy,total_problems,num_success,success_at_k,execution_success_rate,conditional_success
0,phi35mini,humaneval,code_then_plan,164,128,0.7805,0.9512,0.8205
1,phi35mini,humaneval,code_then_plan_repair,164,117,0.7134,0.7378,0.9669
2,phi35mini,humaneval,proposed_v1,164,125,0.7622,0.9268,0.8224
3,phi35mini,humaneval,repair,164,3,0.0183,0.0183,1.0000
4,phi35mini,humaneval,single,164,2,0.0122,0.0366,0.3333
10,qwen25coder7b,humaneval,code_then_plan,164,152,0.9268,0.9939,0.9325
11,qwen25coder7b,humaneval,code_then_plan_repair,164,151,0.9207,0.9756,0.9437
12,qwen25coder7b,humaneval,proposed_v1,164,154,0.9390,0.9939,0.9448
13,qwen25coder7b,humaneval,repair,164,138,0.8415,0.8780,0.9583
14,qwen25coder7b,humaneval,single,164,116,0.7073,0.8841,0.8000


### 1-2. 전략별 피벗 (dataset × model 고정, 전략 비교)

In [5]:
for dataset in df_summary["dataset"].unique():
    sub = df_summary[df_summary["dataset"] == dataset]
    pivot = sub.pivot_table(
        index="model",
        columns="strategy",
        values="success_at_k",
    )
    print(f"\n=== {dataset} | success_at_k ===")
    display(pivot)


=== humaneval | success_at_k ===


strategy,code_then_plan,code_then_plan_repair,proposed_v1,repair,single
model,,,,,
phi35mini,0.7805,0.7134,0.7622,0.0183,0.0122
qwen25coder7b,0.9268,0.9207,0.9390,0.8415,0.7073



=== mbpp | success_at_k ===


strategy,code_then_plan,code_then_plan_repair,proposed_v1,repair,single
model,,,,,
phi35mini,0.7782,0.8054,0.7977,0.6965,0.6342
qwen25coder7b,0.8288,0.8405,0.8599,0.8327,0.5253


### 1-3. 모델별 피벗 (strategy × dataset)

In [6]:
for model in df_summary["model"].unique():
    sub = df_summary[df_summary["model"] == model]
    pivot = sub.pivot_table(
        index="strategy",
        columns="dataset",
        values=["success_at_k", "execution_success_rate", "conditional_success"],
    )
    print(f"\n=== {model} ===")
    display(pivot)


=== phi35mini ===


conditional_success        execution_success_rate  \
dataset                         humaneval   mbpp              humaneval   
strategy                                                                  
code_then_plan                     0.8205 0.8969                 0.9512   
code_then_plan_repair              0.9669 0.9200                 0.7378   
proposed_v1                        0.8224 0.9152                 0.9268   
repair                             1.0000 0.9040                 0.0183   
single                             0.3333 0.8402                 0.0366   

                             success_at_k         
dataset                 mbpp    humaneval   mbpp  
strategy                                          
code_then_plan        0.8677       0.7805 0.7782  
code_then_plan_repair 0.8755       0.7134 0.8054  
proposed_v1           0.8716       0.7622 0.7977  
repair                0.7704       0.0183 0.6965  
single                0.7549       0.0122 0.6342


=== qwen25coder7b ===


conditional_success        execution_success_rate  \
dataset                         humaneval   mbpp              humaneval   
strategy                                                                  
code_then_plan                     0.9325 0.8320                 0.9939   
code_then_plan_repair              0.9437 0.9908                 0.9756   
proposed_v1                        0.9448 0.8735                 0.9939   
repair                             0.9583 0.8327                 0.8780   
single                             0.8000 0.7849                 0.8841   

                             success_at_k         
dataset                 mbpp    humaneval   mbpp  
strategy                                          
code_then_plan        0.9961       0.9268 0.8288  
code_then_plan_repair 0.8482       0.9207 0.8405  
proposed_v1           0.9844       0.9390 0.8599  
repair                1.0000       0.8415 0.8327  
single                0.6693       0.7073 0.5253

## 2. trajectory_logs.jsonl 로드 → 문제별 분석

In [7]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_trajectory_logs(runs: list[dict]) -> pd.DataFrame:
    frames = []
    for r in runs:
        p = r["dir"] / "trajectory_logs.jsonl"
        if not p.exists():
            continue
        rows = load_jsonl(p)
        df = pd.DataFrame(rows)
        df["model"]    = r["model"]
        df["dataset"]  = r["dataset"]
        df["strategy"] = r["strategy"]
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


df_traj = load_trajectory_logs(filtered_runs)
print(f"trajectory rows: {len(df_traj)}")
df_traj.head(3)

trajectory rows: 4210


/tmp/ipykernel_149403/632081574.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


,run_id,dataset,problem_id,method,trajectory_id,num_steps,call_count,final_status,failure_family,final_tests_passed,final_tests_total,total_input_tokens,total_output_tokens,total_tokens,total_latency,num_exec_fail,num_test_fail,transition_path,used_plan,budget_used,recovered_by,plan_recovery_attempt,model,strategy,used_repair,planning_recovery_attempt,planning_cycle_count,planning_cycle_call_cost,repair_call_count,stopped_by_policy,stop_reason,num_plan_calls
0,phase2_phi35mini_humaneval_code_then_plan_0509...,humaneval,HumanEval/0,code_then_plan,HumanEval/0_run0,5,5,PASS,PASS,NaN,None,1389.0000,615.0000,2004,3.8337,2.0000,0.0000,"[EXEC_FAIL:SyntaxError, PLAN_DONE, EXEC_FAIL:S...",True,"{'tokens': 2004, 'calls': 5, 'latency': 3.8336...",plan_code,2.0000,phi35mini,code_then_plan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,phase2_phi35mini_humaneval_code_then_plan_0509...,humaneval,HumanEval/1,code_then_plan,HumanEval/1_run0,5,5,PASS,PASS,NaN,None,1539.0000,833.0000,2372,4.9915,1.0000,1.0000,"[EXEC_FAIL:SyntaxError, PLAN_DONE, TEST_FAIL:A...",True,"{'tokens': 2372, 'calls': 5, 'latency': 4.9914...",plan_code,2.0000,phi35mini,code_then_plan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,phase2_phi35mini_humaneval_code_then_plan_0509...,humaneval,HumanEval/2,code_then_plan,HumanEval/2_run0,5,5,PASS,PASS,NaN,None,1156.0000,909.0000,2065,5.3861,2.0000,0.0000,"[EXEC_FAIL:SyntaxError, PLAN_DONE, EXEC_FAIL:S...",True,"{'tokens': 2065, 'calls': 5, 'latency': 5.3860...",plan_code,2.0000,phi35mini,code_then_plan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2-1. call_count 분포 (전략별 평균 호출 수)

In [8]:
if not df_traj.empty and "call_count" in df_traj.columns:
    display(
        df_traj.groupby(["model", "dataset", "strategy"])["call_count"]
        .agg(["mean", "median", "min", "max", "count"])
        .round(2)
        .sort_index()
    )

mean  median  min  max  count
model         dataset   strategy                                              
phi35mini     humaneval code_then_plan         7.9100  5.0000    1   19    164
                        code_then_plan_repair  9.0500  3.5000    1   19    164
                        proposed_v1            7.4500  4.0000    1   20    164
                        repair                19.6500 20.0000    1   20    164
                        single                 1.0000  1.0000    1    1    164
              mbpp      code_then_plan         5.6100  1.0000    1   19    257
                        code_then_plan_repair  5.3500  1.0000    1   19    257
                        proposed_v1            4.7600  1.0000    1   20    257
                        repair                 6.9200  1.0000    1   20    257
                        single                 1.0000  1.0000    1    1    257
qwen25coder7b humaneval code_then_plan         3.0000  1.0000    1   19    164
                        code_then_plan_repair  2.9700  1.0000    1   19    164
                        proposed_v1            2.4000  1.0000    1   20    164
                        repair                 4.3800  1.0000    1   20    164
                        single                 1.0000  1.0000    1    1    164
              mbpp      code_then_plan         4.8400  1.0000    1   19    257
                        code_then_plan_repair  4.9100  1.0000    1   19    257
                        proposed_v1            4.3900  1.0000    1   20    257
                        repair                 4.5600  1.0000    1   20    257
                        single                 1.0000  1.0000    1    1    257

### 2-2. 문제별 최종 상태 분포

In [9]:
if not df_traj.empty and "final_status" in df_traj.columns:
    status_col = "final_status"
    # coarse failure family
    df_traj["failure_family_coarse"] = df_traj[status_col].apply(
        lambda s: "PASS" if s == "PASS" else str(s).split(":")[0]
    )
    display(
        df_traj.groupby(["model", "dataset", "strategy", "failure_family_coarse"])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["model", "dataset", "strategy", "count"], ascending=[True, True, True, False])
    )

,model,dataset,strategy,failure_family_coarse,count
1,phi35mini,humaneval,code_then_plan,PASS,128
2,phi35mini,humaneval,code_then_plan,TEST_FAIL,28
0,phi35mini,humaneval,code_then_plan,EXEC_FAIL,8
4,phi35mini,humaneval,code_then_plan_repair,PASS,117
3,phi35mini,humaneval,code_then_plan_repair,EXEC_FAIL,43
5,phi35mini,humaneval,code_then_plan_repair,TEST_FAIL,4
7,phi35mini,humaneval,proposed_v1,PASS,125
8,phi35mini,humaneval,proposed_v1,TEST_FAIL,27
6,phi35mini,humaneval,proposed_v1,EXEC_FAIL,12
9,phi35mini,humaneval,repair,EXEC_FAIL,161


### 2-3. 복구 전략 효과 (recovered_by 분포)

In [10]:
if not df_traj.empty and "recovered_by" in df_traj.columns:
    display(
        df_traj.groupby(["model", "dataset", "strategy", "recovered_by"])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["model", "dataset", "strategy", "count"], ascending=[True, True, True, False])
    )

,model,dataset,strategy,recovered_by,count
1,phi35mini,humaneval,code_then_plan,plan_code,124
0,phi35mini,humaneval,code_then_plan,generate,4
3,phi35mini,humaneval,code_then_plan_repair,plan_code,114
2,phi35mini,humaneval,code_then_plan_repair,generate,2
4,phi35mini,humaneval,code_then_plan_repair,repair,1
6,phi35mini,humaneval,proposed_v1,plan_code,78
7,phi35mini,humaneval,proposed_v1,repair,44
5,phi35mini,humaneval,proposed_v1,generate,3
8,phi35mini,mbpp,code_then_plan,generate,163
9,phi35mini,mbpp,code_then_plan,plan_code,37


### 2-4. 초기 실패 → 최종 성공 전환율 (recovery rate)

In [11]:
if not df_traj.empty:
    # initial_failed 컬럼이 없으면 generate step 기준으로 직접 계산
    if "initial_failed" not in df_traj.columns:
        df_traj["initial_failed"] = df_traj.get("transition_path", pd.Series()).apply(
            lambda p: (p[0].split(":")[0] != "PASS") if isinstance(p, list) and p else None
        )

    if "final_status" in df_traj.columns:
        df_traj["final_passed"] = df_traj["final_status"] == "PASS"

    recovery = (
        df_traj[df_traj["initial_failed"] == True]
        .groupby(["model", "dataset", "strategy"])["final_passed"]
        .agg(recovered="sum", total="count")
    )
    recovery["recovery_rate"] = recovery["recovered"] / recovery["total"]
    display(recovery.round(4))

recovered  total  recovery_rate
model         dataset   strategy                                              
phi35mini     humaneval code_then_plan               124    160         0.7750
                        code_then_plan_repair        115    162         0.7099
                        proposed_v1                  122    161         0.7578
                        repair                         0    161         0.0000
                        single                         0    162         0.0000
              mbpp      code_then_plan                37     94         0.3936
                        code_then_plan_repair         42     92         0.4565
                        proposed_v1                   32     84         0.3810
                        repair                        12     90         0.1333
                        single                         0     94         0.0000
qwen25coder7b humaneval code_then_plan                36     48         0.7500
                        code_then_plan_repair         36     49         0.7347
                        proposed_v1                   33     43         0.7674
                        repair                        25     51         0.4902
                        single                         0     48         0.0000
              mbpp      code_then_plan                74    118         0.6271
                        code_then_plan_repair         77    118         0.6525
                        proposed_v1                   89    125         0.7120
                        repair                        74    117         0.6325
                        single                         0    122         0.0000

## 3. step_logs.jsonl 로드 → call 수준 분석

In [12]:
def load_step_logs(runs: list[dict]) -> pd.DataFrame:
    frames = []
    for r in runs:
        p = r["dir"] / "step_logs.jsonl"
        if not p.exists():
            continue
        rows = load_jsonl(p)
        df = pd.DataFrame(rows)
        df["model"]    = r["model"]
        df["dataset"]  = r["dataset"]
        df["strategy"] = r["strategy"]
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


df_steps = load_step_logs(runs)
print(f"step rows: {len(df_steps)}")
df_steps.head(3)

step rows: 42790


/tmp/ipykernel_149403/584948508.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


,run_id,dataset,problem_id,method,trajectory_id,step_id,call_index,candidate_id,stage,is_retry,is_repair,is_planner,policy_action,input_tokens,output_tokens,total_tokens,latency_sec,code,planner_output,exec_ok,test_pass,status,error_type,error_stage,error_message,tests_passed,tests_total,code_length,selected,selection_rank,entry_point,model,strategy,policy_state,policy_stop,policy_stop_reason
0,phase3_llama318b_humaneval_code_then_plan_repa...,humaneval,HumanEval/0,code_then_plan_repair,HumanEval/0_run0,0,0,0,generate,False,False,False,generate,119,512,631,5.5774,None,None,False,False,EXEC_FAIL:SyntaxError,SyntaxError,exec,"Traceback (most recent call last):\n File ""/h...",NaN,None,1371,None,None,has_close_elements,llama318b,code_then_plan_repair,NaN,NaN,NaN
1,phase3_llama318b_humaneval_code_then_plan_repa...,humaneval,HumanEval/0,code_then_plan_repair,HumanEval/0_run0,1,1,0,plan,False,False,True,plan,196,256,452,2.7165,None,None,None,None,PLAN_DONE,None,None,None,NaN,None,0,None,None,has_close_elements,llama318b,code_then_plan_repair,NaN,NaN,NaN
2,phase3_llama318b_humaneval_code_then_plan_repa...,humaneval,HumanEval/0,code_then_plan_repair,HumanEval/0_run0,2,2,0,plan_code,False,False,False,generate,487,512,999,5.4422,None,None,True,True,PASS,None,None,None,NaN,None,1752,None,None,has_close_elements,llama318b,code_then_plan_repair,NaN,NaN,NaN


### 3-1. stage별 PASS 비율 (call 수준 성공률)

In [13]:
if not df_steps.empty and "stage" in df_steps.columns and "status" in df_steps.columns:
    df_steps["passed"] = df_steps["status"] == "PASS"
    stage_stats = (
        df_steps[df_steps["stage"].isin(["generate", "repair", "plan_code"])]
        .groupby(["model", "dataset", "strategy", "stage"])["passed"]
        .agg(pass_count="sum", total="count")
    )
    stage_stats["pass_rate"] = stage_stats["pass_count"] / stage_stats["total"]
    display(stage_stats.round(4))

pass_count  total  \
model         dataset   strategy              stage                          
llama318b     humaneval code_then_plan_repair generate           59    164   
                                              plan_code          39    480   
                                              repair              7    441   
                        proposed_v1           generate           39    164   
                                              plan_code          36    430   
...                                                             ...    ...   
qwen25coder7b mbpp      proposed_v2           plan_code          85    394   
                                              repair              2    244   
                        repair                generate          140    257   
                                              repair             74    915   
                        single                generate          135    257   

                                                         pass_rate  
model         dataset   strategy              stage                 
llama318b     humaneval code_then_plan_repair generate      0.3598  
                                              plan_code     0.0812  
                                              repair        0.0159  
                        proposed_v1           generate      0.2378  
                                              plan_code     0.0837  
...                                                            ...  
qwen25coder7b mbpp      proposed_v2           plan_code     0.2157  
                                              repair        0.0082  
                        repair                generate      0.5447  
                                              repair        0.0809  
                        single                generate      0.5253  

[80 rows x 3 columns]

### 3-2. error_type 분포 (실패 원인 분석)

In [14]:
if not df_steps.empty and "error_type" in df_steps.columns:
    failed_steps = df_steps[
        df_steps["status"].notna()
        & (df_steps["status"] != "PASS")
        & (df_steps["status"] != "PLAN_DONE")
    ]
    display(
        failed_steps.groupby(["model", "dataset", "strategy", "error_type"])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["model", "dataset", "strategy", "count"], ascending=[True, True, True, False])
        .head(60)
    )

,model,dataset,strategy,error_type,count
1,llama318b,humaneval,code_then_plan_repair,IndentationError,389
4,llama318b,humaneval,code_then_plan_repair,SyntaxError,372
0,llama318b,humaneval,code_then_plan_repair,AssertionError,175
3,llama318b,humaneval,code_then_plan_repair,NameError,26
5,llama318b,humaneval,code_then_plan_repair,TypeError,12
2,llama318b,humaneval,code_then_plan_repair,IndexError,5
6,llama318b,humaneval,code_then_plan_repair,ValueError,1
11,llama318b,humaneval,proposed_v1,SyntaxError,490
8,llama318b,humaneval,proposed_v1,IndentationError,325
7,llama318b,humaneval,proposed_v1,AssertionError,185


### 3-3. 토큰 사용량 (step 기준)

In [15]:
if not df_steps.empty and "total_tokens" in df_steps.columns:
    display(
        df_steps.groupby(["model", "dataset", "strategy", "stage"])["total_tokens"]
        .agg(["mean", "median", "sum"])
        .round(1)
        .sort_index()
    )

mean    median  \
model         dataset   strategy              stage                           
llama318b     humaneval code_then_plan_repair generate   643.4000  629.0000   
                                              plan       477.1000  461.0000   
                                              plan_code 1022.0000 1008.0000   
                                              repair    1549.0000 1547.0000   
                        proposed_v1           generate   643.4000  629.0000   
...                                                           ...       ...   
qwen25coder7b mbpp      proposed_v2           repair     855.2000  806.5000   
                                              replan     565.4000  524.0000   
                        repair                generate   202.9000  180.0000   
                                              repair     441.1000  401.0000   
                        single                generate   200.1000  180.0000   

                                                            sum  
model         dataset   strategy              stage              
llama318b     humaneval code_then_plan_repair generate   105522  
                                              plan       228990  
                                              plan_code  490539  
                                              repair     683124  
                        proposed_v1           generate   105522  
...                                                         ...  
qwen25coder7b mbpp      proposed_v2           repair     208661  
                                              replan     155488  
                        repair                generate    52153  
                                              repair     403576  
                        single                generate    51419  

[105 rows x 3 columns]

## 4. 전체 성능 요약 테이블 (논문/보고서용)

In [16]:
REPORT_COLS = [
    "model", "dataset", "strategy",
    "success_at_k", "execution_success_rate", "conditional_success",
    "code_failed", "define_test_failed", "run_test_failed",
]
existing = [c for c in REPORT_COLS if c in df_summary.columns]
df_report = (
    df_summary[existing]
    .sort_values(["dataset", "model", "success_at_k"], ascending=[True, True, False])
    .reset_index(drop=True)
)
display(df_report)

,model,dataset,strategy,success_at_k,execution_success_rate,conditional_success,code_failed,define_test_failed,run_test_failed
0,phi35mini,humaneval,code_then_plan,0.7805,0.9512,0.8205,0,0,0
1,phi35mini,humaneval,proposed_v1,0.7622,0.9268,0.8224,12,0,27
2,phi35mini,humaneval,code_then_plan_repair,0.7134,0.7378,0.9669,0,0,0
3,phi35mini,humaneval,repair,0.0183,0.0183,1.0000,161,0,0
4,phi35mini,humaneval,single,0.0122,0.0366,0.3333,158,0,4
5,qwen25coder7b,humaneval,proposed_v1,0.9390,0.9939,0.9448,1,0,9
6,qwen25coder7b,humaneval,code_then_plan,0.9268,0.9939,0.9325,0,0,0
7,qwen25coder7b,humaneval,code_then_plan_repair,0.9207,0.9756,0.9437,0,0,0
8,qwen25coder7b,humaneval,repair,0.8415,0.8780,0.9583,20,0,6
9,qwen25coder7b,humaneval,single,0.7073,0.8841,0.8000,19,0,29


In [17]:
# CSV로 저장
out_path = Path("results_report.csv")
df_report.to_csv(out_path, index=False)
print(f"저장 완료: {out_path.resolve()}")

저장 완료: /home/dibaeck/workspace/project_IR_focus_sLM_orchestration/analysis/results_report.csv
